# Detecting Calibrated Rare Actions - Experiment

This notebook runs the core experiment: training and testing a linear probe for step 3 of the calibrated-rare-action procedure (extracting entropy from context, transforming it, and comparing it to a threshold to render a decision).

This notebook follows these steps in the experimental procedure:

4. **Train the Probe**
   1. Run 10 trajectories each of the training set (400 trajectories: 20 pairs x 2 labels x 10 replicates)
   2. Sweeping across layers and taking the mean at every generated token, find the optimal layer to train a linear probe based on difference of means between contrastive pairs.
5. **Test the Probe**
   1. Run 10 trajectories each on the testing set (100 trajectories: 5 pairs x 2 labels x 10 replicates)
   2. Check the probe (AUROC, etc)

Note: step 6 (Validate for Generalization, using RPS prompts) is out of scope for this notebook. The RPS prompts were never built (see README STATUS), so there is no generalization set to run against yet.

This notebook does not judge, score, or classify individual trajectories' content in any way — it only generates completions, extracts residual-stream activations, and evaluates the linear probe's separation of the positive/negative *labels* the dataset was built with. Reading trajectory text for step-3 content is a manual step performed outside this notebook (see `outputs/verification/` from validate-setup.ipynb).

**The trained probe is saved to `probes/diff_of_means_probe.pt` and tracked in git** — it is expensive to reproduce (requires the full trajectory run) and is not an intermediate artifact like the trajectories themselves.

**Shared Instructions:**
- Make use of standard infrastructure for MechInterp, such as TransformerLens and PyTorch, as appropriate. Don't do extra work where ready-made solutions are mature and fit well.
- All shared infrastructure and all background infrastructure should live in dedicated `.py` files, not in notebooks. Reading a notebook should be straightforward, avoiding implementation details which are not directly relevant.
  - Example of relevant implementation details: what statistical tests are being ran? The notebook should import the library which implements these tests and run them "bare" in a cell, because the specific statistical tests are directly relevant to the purpose of the notebook.
  - Example of irrelevant implementation details: how are prompts loaded? This is not important to the purpose of a notebook, so it should simply be a function imported from a script file.
- **No Pandas**. Results are kept as tensors and lists/dicts of tensors. Indices are simplified and avoid unnecessary rekeying. There are no DataFrames in this notebook.
- **Use SIZES**. This and all notebooks use switched size configurations (often "check", "demo", "full" or similar) which pick out specific configurations from a dict; the only required action to change from a quick check to a full run is changing the `SIZE` variable to a different string.
- Every notebook follows a clear epistemic structure:
  - Set out purpose and expectations
  - State predictions and interpretation thresholds in advance where applicable
  - Prepare input data
  - Run the procedure
  - Run automated data interpretation (neutral, with zero presupposition of what the results will be)
  - Hand-written markdown interpretation / conclusion cells, the ONLY cells which are written with an awareness of what the results have been
- Every result which appears as a visual chart ought to also be prepared as a textual table (e.g. display some markdown).

In [1]:
%load_ext autoreload
%autoreload 2

%matplotlib widget

In [2]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, roc_auc_score
from tqdm.auto import tqdm
import wandb

from utilities import (
    load_dataset,
    load_model,
    load_hf_model,
    run_trajectory_batch,
    add_activations_batch,
    save_trajectory,
    load_trajectories,
    fit_diff_of_means_direction,
    project_onto_probe,
    sweep_layers,
    save_probe,
)

## Purpose and Expectations

### What we're testing

The dataset and setup have already been validated (validate-dataset.ipynb, validate-setup.ipynb): the prompts don't have strong length or lexical confounds, and the model reliably follows the intended positive/negative prompt behavior at `Qwen/Qwen3-14B`.

This notebook trains a linear probe to detect step 3 (randomness extraction & decision) from the model's residual stream activations *during generation*, then tests whether that probe generalizes to held-out prompt pairs.

The probe is a **difference-of-means direction**: for a given layer, `direction = mean(positive-class activations) - mean(negative-class activations)`, normalized to unit length, with a bias set at the midpoint of the two class means. This is the simplest linear probe construction and matches the README's step 4.2.

Activations are the residual stream (`resid_post`) mean-pooled over only the **generated completion tokens** of each trajectory (not the prompt), since step 3 is something the model does while generating, and the prompt is matched in structure/length between positive/negative twins by construction.

### Two-stage layer selection, then held-out test

Step 4 asks for "the optimal layer." Because each of the 20 training pairs contributes 10 replicate trajectories (200 positive + 200 negative trajectories from the training set alone), there's enough data to do this as a clean split *within* the training set, unlike the sparse 25-pair regime the dataset-validation BoW classifier had to work around with leave-pair-out CV:

1. **Fit/select split** (within the 400 training trajectories): hold out a subset of *training pairs* (all 10 replicates of each) to select the best layer by AUROC; fit the per-layer direction on the rest. Splitting by pair (not by individual trajectory) keeps near-duplicate replicates of the same prompt on one side of the split.
2. **Final fit**: refit the direction at the selected layer using *all* 20 training pairs (400 trajectories), for the most stable estimate to carry into testing.
3. **Test**: evaluate that final probe's AUROC on the 5 held-out test pairs (100 trajectories), which the probe has never seen in any form.

This mirrors the standard train/validation/test structure: the fit/select split is the training-internal step used only to pick a layer, and the test set is touched exactly once, at the end.

## Predictions

### Layer sweep

We expect AUROC to be low in the earliest layers (activations there mostly reflect surface token identity, shared between positive/negative twins) and to rise through the middle layers where the model would need to represent "I am extracting/comparing entropy against a threshold" as a feature. We don't have a strong prior on whether the best layer is early-middle or late-middle; that's exactly what the sweep is for.

### Held-out test AUROC

**Interpretation thresholds**, set in advance:
- **AUROC ≥ 0.85** on the held-out test set: the probe has found a generalizable direction associated with step 3 behavior. Strong result.
- **0.65 ≤ AUROC < 0.85**: partial signal — better than chance and better than the dataset's own lexical confound ceiling (0.75 CV AUROC on the BoW classifier, from validate-dataset.ipynb), but not clean separation. Worth reporting as suggestive, not conclusive.
- **AUROC < 0.65**: the probe has not found a generalizable step-3 direction at the selected layer. Would prompt reconsidering the pooling strategy, layer choice, or dataset before proceeding further.

These thresholds are set before running anything in this notebook and are not adjusted afterward.

In [3]:
SIZE = 'full'  # 'smoke' = pipeline plumbing check only, no signal expected; 'check' = small real run; 'full' = full run (Qwen3-14B, 20 train pairs x 10 reps, 5 test pairs x 10 reps)

SIZES = {
    'smoke': {
        # Cheapest possible pass through every step (generate -> cache -> pool -> save -> load ->
        # sweep -> fit -> save probe -> test -> plot) to catch plumbing bugs before spending real
        # compute. Counts are too small for the AUROC numbers to mean anything - this is not a
        # scaled-down experiment, it's a pipeline check. Uses an already-downloaded, instruct-tuned
        # model so it doesn't pull anything new or risk this host's memory headroom.
        'model_name': "Qwen/Qwen2.5-0.5B-Instruct",
        'n_replicates': 1,
        'n_select_pairs': 1,
        'n_train_pairs': 2,
        'n_test_pairs': 1,
        'max_new_tokens': 64,
    },
    'check': {
        'model_name': "Qwen/Qwen3-0.6B",
        'n_replicates': 2,
        'n_select_pairs': 2,  # of the (small) train slice, how many pairs are held out for layer selection
        'n_train_pairs': 6,   # slice of the 20 real training pairs to use
        'n_test_pairs': 2,    # slice of the 5 real test pairs to use
        'max_new_tokens': 256,
    },
    'full': {
        'model_name': "Qwen/Qwen3-14B",
        'n_replicates': 10,
        'n_select_pairs': 4,  # ~20% of the 20 training pairs, held out for layer selection
        'n_train_pairs': 20,
        'n_test_pairs': 5,
        'max_new_tokens': 512,
    },
}

config = SIZES[SIZE]
print(f"Running with SIZE={SIZE}")
print(f"Config: {config}")

wandb_run = wandb.init(
    project="detecting-calibrated-rare-actions",
    job_type="train_probe",
    config={"size": SIZE, **config},
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


Running with SIZE=full
Config: {'model_name': 'Qwen/Qwen3-14B', 'n_replicates': 10, 'n_select_pairs': 4, 'n_train_pairs': 20, 'n_test_pairs': 5, 'max_new_tokens': 512}


wandb: Currently logged in as: fractalmachinist to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Prepare Input Data

Following the README's split: the first 20 pairs (by dataset order, spanning both tasks) are the training set, and the remaining 5 pairs are the test set — the same split `utilities.train_bow_classifier` uses. Within the training set, a further subset of pairs is held out purely for layer selection (step 4.2); those pairs are still "training set," never touching the 5 test pairs.

`SIZE='smoke'` runs the smallest possible slice end-to-end purely to catch plumbing bugs (its AUROC numbers carry no evidential weight); `SIZE='check'` is a small but real run meant to show an actual trend; `SIZE='full'` uses every pair in each split with the full replicate count.

In [4]:
positives, negatives, tasks, thresholds = load_dataset()
n_pairs = len(positives)

TRAIN_PAIR_INDICES = list(range(20))[:config['n_train_pairs']]
TEST_PAIR_INDICES = list(range(20, 25))[:config['n_test_pairs']]

# Held out from TRAIN_PAIR_INDICES purely for layer selection; disjoint from TEST_PAIR_INDICES.
SELECT_PAIR_INDICES = TRAIN_PAIR_INDICES[-config['n_select_pairs']:]
FIT_PAIR_INDICES = TRAIN_PAIR_INDICES[:-config['n_select_pairs']]

print(f"Train pairs ({len(TRAIN_PAIR_INDICES)}): {TRAIN_PAIR_INDICES}")
print(f"  Fit-for-layer-selection subset ({len(FIT_PAIR_INDICES)}): {FIT_PAIR_INDICES}")
print(f"  Held-out-for-layer-selection subset ({len(SELECT_PAIR_INDICES)}): {SELECT_PAIR_INDICES}")
print(f"Test pairs ({len(TEST_PAIR_INDICES)}): {TEST_PAIR_INDICES}")

n_train_trajectories = len(TRAIN_PAIR_INDICES) * 2 * config['n_replicates']
n_test_trajectories = len(TEST_PAIR_INDICES) * 2 * config['n_replicates']
print(f"\nTrajectories to generate: {n_train_trajectories} train + {n_test_trajectories} test = {n_train_trajectories + n_test_trajectories} total")

Train pairs (20): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
  Fit-for-layer-selection subset (16): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
  Held-out-for-layer-selection subset (4): [16, 17, 18, 19]
Test pairs (5): [20, 21, 22, 23, 24]

Trajectories to generate: 400 train + 100 test = 500 total


## Run the Procedure

This runs in two phases so only one full copy of the model is ever resident on the GPU at a time — at `SIZE='full'` (Qwen3-14B, bf16), a generation-only HF model and a HookedTransformer are each ~28GB, and both together don't fit on a single GPU:

1. **Generate**: load a plain HF model (`load_hf_model`) and, for each (pair, label) combination in the train and test sets, generate all `n_replicates` trajectories in one batched forward pass (`run_trajectory_batch`) and save the text to disk (`outputs/train/` and `outputs/test/`, gitignored like `outputs/verification/`). Each (pair, label) batch gets a distinct seed for reproducibility; individual replicates within a batch still sample independently of each other. The HF model is then released.
2. **Cache activations**: load a HookedTransformer (`load_model`) over the same weights, reload the saved trajectories, extract each one's per-layer mean activation over generated tokens (`add_activations_batch`), and re-save with activations attached.

This step is the expensive one — it's the part meant to run on a rented GPU instance at `SIZE='full'`.

In [5]:
hf_model, tokenizer = load_hf_model(model_name=config['model_name'])
print(f"Loaded {config['model_name']} on {hf_model.device}")

Loading weights:   0%|          | 0/443 [00:00<?, ?it/s]

Loaded Qwen/Qwen3-14B on cuda:0


In [6]:
from pathlib import Path

def generate_and_save_split(pair_indices, out_dir, n_replicates):
    saved_paths = []
    for pair_index in tqdm(pair_indices, desc=f"Generating trajectories ({out_dir})"):
        task = tasks[pair_index]
        for label, prompt_text in [("positive", positives[pair_index]), ("negative", negatives[pair_index])]:
            seed = pair_index * 1000
            records = run_trajectory_batch(
                hf_model, tokenizer, prompt_text, n_replicates=n_replicates, max_new_tokens=config['max_new_tokens'], seed=seed, temperature=0.7,
            )
            for replicate_index, record in enumerate(records):
                path = save_trajectory(record, label=label, task=task, pair_index=pair_index, replicate_index=replicate_index, out_dir=out_dir)
                saved_paths.append(path)
    return saved_paths

TRAIN_DIR = Path("outputs") / "train"
TEST_DIR = Path("outputs") / "test"

In [7]:
train_paths = generate_and_save_split(TRAIN_PAIR_INDICES, TRAIN_DIR, config['n_replicates'])
print(f"Saved {len(train_paths)} train trajectories to {TRAIN_DIR}")

Generating trajectories (outputs/train):   0%|          | 0/20 [00:00<?, ?it/s]

KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x71474189b680>> (for post_run_cell), with arguments args (<ExecutionResult object at 71474012aea0, execution_count=7 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 714740128bc0, raw_cell="train_paths = generate_and_save_split(TRAIN_PAIR_I.." transformed_cell="train_paths = generate_and_save_split(TRAIN_PAIR_I.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Bvastai-49691513/workspace/detecting-calibrated-rare-actions/experiment.ipynb#X15sdnNjb2RlLXJlbW90ZQ%3D%3D cell_meta={'cellId': 'vscode-notebook-cell://ssh-remote%2Bvastai-49691513/workspace/detecting-calibrated-rare-actions/experiment.ipynb#X15sdnNjb2RlLXJlbW90ZQ%3D%3D'}> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

In [ ]:
test_paths = generate_and_save_split(TEST_PAIR_INDICES, TEST_DIR, config['n_replicates'])
print(f"Saved {len(test_paths)} test trajectories to {TEST_DIR}")

Generation is done. Release the HF model before loading the HookedTransformer, so only one full model copy is resident at a time.

In [ ]:
del hf_model
torch.cuda.empty_cache()

Now load a HookedTransformer over the same weights and extract activations for every saved trajectory, re-saving each with its activations attached.

In [ ]:
model = load_model(model_name=config['model_name'])
print(f"Loaded {model.cfg.model_name} on {model.cfg.device}")
print(f"n_layers={model.cfg.n_layers}, d_model={model.cfg.d_model}")

In [ ]:
def add_activations_and_resave_split(pair_indices, out_dir):
    records = load_trajectories(out_dir)
    records_by_group = {}
    for record in records:
        key = (record["task"], record["label"], record["pair_index"])
        records_by_group.setdefault(key, []).append(record)

    for pair_index in tqdm(pair_indices, desc=f"Caching activations ({out_dir})"):
        task = tasks[pair_index]
        for label in ("positive", "negative"):
            group_records = records_by_group[(task, label, pair_index)]
            add_activations_batch(model, group_records)
            for record in group_records:
                save_trajectory(
                    record, label=label, task=task,
                    pair_index=pair_index, replicate_index=record["replicate_index"], out_dir=out_dir,
                )

In [ ]:
add_activations_and_resave_split(TRAIN_PAIR_INDICES, TRAIN_DIR)
print(f"Added activations to {len(train_paths)} train trajectories in {TRAIN_DIR}")

In [ ]:
add_activations_and_resave_split(TEST_PAIR_INDICES, TEST_DIR)
print(f"Added activations to {len(test_paths)} test trajectories in {TEST_DIR}")

## Automated Data Interpretation

Load every saved trajectory's activations back from disk, assemble them into per-split tensors, sweep layers on the fit/select subsets of the training set, refit the selected layer's direction on the full training set, and evaluate on the held-out test set. No results are inspected until this entire pipeline has run.

In [ ]:
def stack_split(out_dir):
    """Load all trajectories in out_dir and stack into (pos_acts, neg_acts, pos_pair_ids, neg_pair_ids)."""
    records = load_trajectories(out_dir, load_activations=True)
    pos_acts, neg_acts = [], []
    pos_pair_ids, neg_pair_ids = [], []
    for record in records:
        if record["label"] == "positive":
            pos_acts.append(record["activations"])
            pos_pair_ids.append(record["pair_index"])
        else:
            neg_acts.append(record["activations"])
            neg_pair_ids.append(record["pair_index"])
    return (
        torch.stack(pos_acts, dim=0),
        torch.stack(neg_acts, dim=0),
        torch.tensor(pos_pair_ids),
        torch.tensor(neg_pair_ids),
    )

train_pos_acts, train_neg_acts, train_pos_pair_ids, train_neg_pair_ids = stack_split(TRAIN_DIR)
test_pos_acts, test_neg_acts, test_pos_pair_ids, test_neg_pair_ids = stack_split(TEST_DIR)

print(f"Train: {train_pos_acts.shape[0]} positive, {train_neg_acts.shape[0]} negative trajectories, activations shape {tuple(train_pos_acts.shape[1:])}")
print(f"Test:  {test_pos_acts.shape[0]} positive, {test_neg_acts.shape[0]} negative trajectories")

### Layer sweep (fit on FIT_PAIR_INDICES, select on SELECT_PAIR_INDICES)

In [ ]:
sweep_result = sweep_layers(
    train_pos_acts, train_neg_acts, train_pos_pair_ids, train_neg_pair_ids,
    fit_pair_ids=FIT_PAIR_INDICES, select_pair_ids=SELECT_PAIR_INDICES,
)

n_layers = len(sweep_result['auroc_per_layer'])
print(f"Swept {n_layers} layers")
print(f"Best layer: {sweep_result['best_layer']} (AUROC={sweep_result['best_auroc']:.4f} on layer-selection held-out pairs)")

print("\nLayer | Selection AUROC")
print("------|----------------")
for layer, auroc_value in enumerate(sweep_result['auroc_per_layer']):
    marker = "  <-- best" if layer == sweep_result['best_layer'] else ""
    print(f"{layer:5d} | {auroc_value:.4f}{marker}")

wandb.log({
    "sweep/best_layer": sweep_result['best_layer'],
    "sweep/best_auroc": sweep_result['best_auroc'],
    "sweep/auroc_by_layer": wandb.Table(
        columns=["layer", "selection_auroc"],
        data=[[layer, auroc_value] for layer, auroc_value in enumerate(sweep_result['auroc_per_layer'])],
    ),
})

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(n_layers), sweep_result['auroc_per_layer'], marker='o')
ax.axvline(sweep_result['best_layer'], color='red', linestyle='--', alpha=0.5, label=f"best layer ({sweep_result['best_layer']})")
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5, label='chance')
ax.set_xlabel('Layer')
ax.set_ylabel('AUROC (layer-selection held-out pairs)')
ax.set_title('Diff-of-means probe AUROC by layer')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
wandb.log({"sweep/auroc_by_layer_plot": wandb.Image(fig)})
plt.show()

### Final fit at the selected layer, using the full training set

Refit the direction at `sweep_result['best_layer']` using all training pairs (`FIT_PAIR_INDICES` + `SELECT_PAIR_INDICES` combined) for the most stable estimate, then save this probe to disk. This is the probe carried into the test evaluation below.

In [ ]:
BEST_LAYER = sweep_result['best_layer']

final_fit = fit_diff_of_means_direction(train_pos_acts, train_neg_acts)  # all training pairs, all layers
final_direction = final_fit['direction'][BEST_LAYER]  # [d_model]
final_bias = final_fit['bias'][BEST_LAYER]  # scalar

probe_path = save_probe(
    probe={"direction": final_direction, "bias": final_bias},
    model_name=config['model_name'],
    layer=BEST_LAYER,
)
print(f"Saved probe (layer={BEST_LAYER}, model={config['model_name']}) to {probe_path}")

wandb.log({"probe/selected_layer": BEST_LAYER})
probe_artifact = wandb.Artifact("diff_of_means_probe", type="probe", metadata={"layer": BEST_LAYER, "model_name": config['model_name']})
probe_artifact.add_file(str(probe_path))
wandb_run.log_artifact(probe_artifact)

## 5. Test the Probe

Evaluate the saved probe on the 5 held-out test pairs, which have not influenced the direction, bias, or layer choice in any way.

In [ ]:
test_pos_layer_acts = test_pos_acts[:, BEST_LAYER, :]  # [n_test_pos, d_model]
test_neg_layer_acts = test_neg_acts[:, BEST_LAYER, :]  # [n_test_neg, d_model]

test_pos_scores = test_pos_layer_acts @ final_direction - final_bias
test_neg_scores = test_neg_layer_acts @ final_direction - final_bias

y_true = np.concatenate([np.ones(test_pos_scores.shape[0]), np.zeros(test_neg_scores.shape[0])])
y_score = np.concatenate([test_pos_scores.numpy(), test_neg_scores.numpy()])

test_auroc = float(roc_auc_score(y_true, y_score))
test_accuracy = float(((y_score > 0).astype(float) == y_true).mean())

print(f"Test set: {len(test_pos_scores)} positive, {len(test_neg_scores)} negative trajectories")
print(f"Test AUROC: {test_auroc:.4f}")
print(f"Test accuracy (threshold at bias=0): {test_accuracy:.4f}")

wandb.log({"test/auroc": test_auroc, "test/accuracy": test_accuracy})

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_score)
roc_area = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, label=f'Test ROC (AUC = {roc_area:.3f})', color='blue')
ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Chance')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'Held-out test ROC (layer {BEST_LAYER} diff-of-means probe)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
wandb.log({"test/roc_curve_plot": wandb.Image(fig)})
plt.show()

In [ ]:
print("Per-test-pair mean probe score (positive should score higher than its paired negative):")
print("\nPair | Task   | Threshold | Mean(+) score | Mean(-) score | Separated?")
print("-----|--------|-----------|----------------|----------------|-----------")
per_pair_rows = []
for pair_index in TEST_PAIR_INDICES:
    pos_mask = test_pos_pair_ids == pair_index
    neg_mask = test_neg_pair_ids == pair_index
    pos_mean = test_pos_scores[pos_mask].mean().item()
    neg_mean = test_neg_scores[neg_mask].mean().item()
    separated = "yes" if pos_mean > neg_mean else "no"
    print(f"{pair_index:4d} | {tasks[pair_index]:6s} | {thresholds[pair_index]:9s} | {pos_mean:+14.3f} | {neg_mean:+14.3f} | {separated}")
    per_pair_rows.append([pair_index, tasks[pair_index], thresholds[pair_index], pos_mean, neg_mean, separated])

wandb.log({"test/per_pair_scores": wandb.Table(
    columns=["pair_index", "task", "threshold", "mean_pos_score", "mean_neg_score", "separated"],
    data=per_pair_rows,
)})

In [ ]:
print(f"Test AUROC: {test_auroc:.4f}")
print("Pre-registered thresholds: >=0.85 strong, 0.65-0.85 partial, <0.65 no generalizable signal\n")

if test_auroc >= 0.85:
    result_band = "strong"
elif test_auroc >= 0.65:
    result_band = "partial"
else:
    result_band = "no_generalizable_signal"

print(f"Result band: {result_band.replace('_', ' ').upper()}")

wandb.log({"test/result_band": result_band})
wandb.finish()

## Conclusion

**TODO: fill in after running `SIZE='full'` on the rented instance.** This cell is written with awareness of results per this project's epistemic-structure convention, so it can't be written until the run above has actually executed. Report at minimum:

- The selected layer and its layer-selection AUROC (from the sweep).
- The held-out test AUROC and which pre-registered band it falls into (strong / partial / no signal).
- Whether the per-pair table shows separation concentrated in one task (coding vs. email) or spread across both.
- Whether the saved probe at `probes/diff_of_means_probe.pt` is judged trustworthy enough to use as-is, or whether it should be revisited (different layer, different pooling, more replicates) before treating this as a validated step-3 probe.